## 1 Setup & Daten laden

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, HDBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.neighbors import NearestNeighbors

RANDOM_STATE = 20
SIL_SAMPLE   = 3000   # Stichprobe für Silhouette (O(n^2) -> auf 21k zu teuer)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

In [ ]:
df = pd.read_csv("cleaned_house_data_new.csv")

clustering_cols = [c for c in df.columns if not c.startswith('prof_')]
X = df[clustering_cols].values                                    # nur die 6 Cluster-Features
profile = df[[c for c in df.columns if c.startswith('prof_')]]    # prof_price/waterfront/grade

print("Clustering auf:", clustering_cols)
print("Profil-Spalten:", list(profile.columns))
print("Shape gesamt:", df.shape, "| X:", X.shape)               # X sollte (21420, 6) sein
print("Fehlende Werte gesamt:", int(df.isna().sum().sum()))
df.head()

In [ ]:
# Kurzer Skalen-Check: sollte Mittel ~0 und Std ~1 sein (aus M3)
df[clustering_cols].describe().T[["mean", "std", "min", "max"]].round(3)

## 2  Kurze EDA: Verteilungen & Korrelationen

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.ravel(), clustering_cols):
    sns.histplot(df[col], bins=40, kde=True, ax=ax, color="#4C72B0")
    ax.set_title(col, fontsize=9)
    ax.set_xlabel("")
fig.suptitle("Verteilungen der standardisierten Cluster-Features", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(7, 5.5))
sns.heatmap(df[clustering_cols].corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, cbar_kws={"shrink": .8})
plt.title("Korrelationsmatrix der Features")
plt.tight_layout(); plt.show()

## 3 Wie viele Cluster? (Elbow · Silhouette · Davies-Bouldin)
Wir bestimmen $k$ **empirisch** mit K-Means über $k = 2 \dots 14$ und drei Kennzahlen:

- **SSE / Inertia (Elbow):** Knick im Verlauf → guter $k$-Kandidat.
- **Silhouette:** je höher, desto besser getrennt (Wertebereich −1…+1).
- **Davies-Bouldin:** je **niedriger**, desto besser getrennt (0 = ideal).


In [ ]:
k_range = range(2, 9)
sse, sil, dbi = [], [], []

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
    labels = km.labels_
    sse.append(km.inertia_)
    sil.append(silhouette_score(X, labels, sample_size=SIL_SAMPLE, random_state=RANDOM_STATE))
    dbi.append(davies_bouldin_score(X, labels))

metrics = pd.DataFrame({"k": list(k_range),
                        "SSE": sse,
                        "Silhouette": sil,
                        "Davies_Bouldin": dbi}).set_index("k")

# Anzeige mit k waagerecht (Spalten = k, Zeilen = Kennzahlen); metrics selbst bleibt
# unverändert, damit die Plots und idxmax()/idxmin() unten k weiter als Index nutzen.
metrics.T.round(3)

In [ ]:
best_sil = metrics["Silhouette"].idxmax()
best_dbi = metrics["Davies_Bouldin"].idxmin()

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(list(k_range), sse, "o-", color="#4C72B0"); ax[0].set_title("Elbow (SSE) – Knick suchen")
ax[1].plot(list(k_range), sil, "o-", color="#55A868"); ax[1].set_title("Silhouette – höher = besser")
ax[2].plot(list(k_range), dbi, "o-", color="#C44E52"); ax[2].set_title("Davies-Bouldin – niedriger = besser")

# Silhouette-Optimum markieren
ax[1].axvline(best_sil, ls="--", c="grey", alpha=.7)
ax[1].scatter([best_sil], [metrics.loc[best_sil, "Silhouette"]], color="black", zorder=5)
ax[1].annotate(f"max @ k={best_sil}", (best_sil, metrics.loc[best_sil, "Silhouette"]),
               textcoords="offset points", xytext=(6, -10), fontsize=8)

# Davies-Bouldin-Optimum markieren
ax[2].axvline(best_dbi, ls="--", c="grey", alpha=.7)
ax[2].scatter([best_dbi], [metrics.loc[best_dbi, "Davies_Bouldin"]], color="black", zorder=5)
ax[2].annotate(f"min @ k={best_dbi}", (best_dbi, metrics.loc[best_dbi, "Davies_Bouldin"]),
               textcoords="offset points", xytext=(6, 6), fontsize=8)

for a in ax:
    a.set_xlabel("k")
plt.tight_layout(); plt.show()

print(f"Bestes k nach Silhouette:     {best_sil}")
print(f"Bestes k nach Davies-Bouldin: {best_dbi}")

## 5 Algorithmen-Vergleich
Für jeden Algorithmus: fitten → Cluster im PCA-Raum plotten → Kennzahlen sammeln. Density-Verfahren (DBSCAN/HDBSCAN) bestimmen die Clusterzahl selbst und markieren Ausreißer als **Noise** (Label −1).

In [ ]:
from matplotlib.lines import Line2D

results = []   # Sammel-Liste für die finale Vergleichstabelle

def evaluate(name, labels):
    # Kennzahlen robust berechnen (Noise -1 ausschliessen, >=2 Cluster noetig)
    labels = np.asarray(labels)
    mask = labels != -1
    uniq = set(labels[mask])
    n_clusters = len(uniq)
    n_noise = int((labels == -1).sum())
    if n_clusters >= 2 and mask.sum() > n_clusters:
        sil = silhouette_score(X[mask], labels[mask],
                               sample_size=min(SIL_SAMPLE, mask.sum()),
                               random_state=RANDOM_STATE)
        dbi = davies_bouldin_score(X[mask], labels[mask])
    else:
        sil, dbi = np.nan, np.nan
    results.append({"Algorithmus": name, "n_Cluster": n_clusters,
                    "Noise": n_noise, "Silhouette": sil, "Davies_Bouldin": dbi})
    print(f"{name:16s} | Cluster: {n_clusters:2d} | Noise: {n_noise:5d} "
          f"| Silhouette: {sil:.3f} | DB: {dbi:.3f}")
    return labels

def plot_clusters(labels, title, emb=None, idx=None, ax=None,
                  xlabel="PC1", ylabel="PC2"):
    """Cluster in einer 2D-Projektion – feste Farbe pro Cluster, beschriftetes
    Zentrum und Legende (inkl. Größe), damit die Segmente klar erkennbar sind.

    emb : 2D-Koordinaten (default: X_pca). idx : optionale Punkt-Auswahl
    (z. B. t-SNE-Stichprobe) – dann werden labels[idx] geplottet.
    ax  : zum Einbetten in ein Grid; ohne ax wird eine eigene Figur erzeugt.
    """
    labels = np.asarray(labels)
    if emb is None:
        emb = X_pca
    if idx is not None:
        labels = labels[idx]
    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(7.5, 6))

    # Cluster nach Größe: große zuerst zeichnen, kleine kommen oben drauf -> sichtbar
    clusters = sorted((c for c in np.unique(labels) if c != -1),
                      key=lambda c: (labels == c).sum(), reverse=True)
    cmap = plt.get_cmap("tab10" if len(clusters) <= 10 else "tab20")
    colors = {c: cmap(i % cmap.N) for i, c in enumerate(clusters)}

    # Noise dezent in den Hintergrund
    noise = labels == -1
    if noise.any():
        ax.scatter(emb[noise, 0], emb[noise, 1], c="lightgrey",
                   s=6, alpha=.30, linewidths=0, zorder=1)

    handles = []
    for i, c in enumerate(clusters):
        m = labels == c
        ax.scatter(emb[m, 0], emb[m, 1], color=colors[c],
                   s=14, alpha=.65, linewidths=0, zorder=2 + i)
        # Cluster-Zentrum hervorheben + beschriften (nur als Anker fürs Label)
        cx, cy = emb[m, 0].mean(), emb[m, 1].mean()
        ax.scatter(cx, cy, marker="o", s=280, color=colors[c],
                   edgecolor="black", linewidths=1.8, zorder=50)
        ax.text(cx, cy, str(c), color="black", fontsize=10, fontweight="bold",
                ha="center", va="center", zorder=51)
        handles.append(Line2D([0], [0], marker="o", linestyle="", markersize=9,
                              markerfacecolor=colors[c], markeredgecolor="black",
                              label=f"Cluster {c}  (n={int(m.sum()):,})"))
    if noise.any():
        handles.append(Line2D([0], [0], marker="o", linestyle="", markersize=9,
                              markerfacecolor="lightgrey", markeredgecolor="none",
                              label=f"Noise  (n={int(noise.sum()):,})"))

    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(handles=handles, loc="best", framealpha=.9, fontsize=8)
    if own_fig:
        plt.tight_layout(); plt.show()

### 5.1 · K-Means

In [ ]:
# 2D-Projektion EINMAL berechnen -> alle Panels vergleichbar
X_pca = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X)

ks = list(range(2, 9))
ncols = 3
nrows = int(np.ceil(len(ks) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.6 * nrows))
axes = axes.ravel()

for ax, k in zip(axes, ks):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
    sil = silhouette_score(X, km.labels_,
                           sample_size=min(SIL_SAMPLE, len(X)),
                           random_state=RANDOM_STATE)
    plot_clusters(km.labels_, f"k={k}  (Sil={sil:.3f})", emb=X_pca, ax=ax)

for ax in axes[len(ks):]:
    ax.set_visible(False)

fig.suptitle("K-Means auf X (PCA-2D-Projektion), k = 2 … 11",
             fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

### 5.2 · Hierarchical (Agglomerative, Ward)
Dendrogramm auf einer Zufalls-Stichprobe (Ward/Linkage ist $O(n^2)$ Speicher – für 21k unpraktisch, zur *Visualisierung* ist eine Stichprobe Standard. Das eigentliche Clustering fitten wir mit `AgglomerativeClustering` auf allen Punkten.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
idx = rng.choice(len(X), size=2000, replace=False)
Z = linkage(X[idx], method="ward")

plt.figure(figsize=(11, 4))
dendrogram(Z, truncate_mode="level", p=5, no_labels=True, color_threshold=None)
plt.title("Dendrogramm (Ward, Stichprobe n=2000)")
plt.xlabel("Beispiele"); plt.ylabel("Ward-Distanz")
plt.tight_layout(); plt.show()

In [ ]:
# Bevorzugt Ward selbst ein anderes k? (der Scan in §4 lief nur mit K-Means)
# Ward-Linkage EINMAL auf allen Punkten berechnen, dann je k schneiden (effizient).
from scipy.cluster.hierarchy import fcluster

Z_full = linkage(X, method="ward")          # O(n^2) Speicher, auf 21k noch machbar
ward_k = range(2, 9)
w_sil, w_dbi = [], []
for k in ward_k:
    lab = fcluster(Z_full, t=k, criterion="maxclust")
    w_sil.append(silhouette_score(X, lab, sample_size=SIL_SAMPLE, random_state=RANDOM_STATE))
    w_dbi.append(davies_bouldin_score(X, lab))

# Ward-"Elbow": Ward hat kein SSE/Inertia wie K-Means. Das hierarchische Pendant sind
# die MERGE-DISTANZEN (Höhen im Dendrogramm) – die Distanz, bei der die Daten in k
# Cluster zerfallen. Ein Knick markiert (wie der SSE-Elbow) eine natürliche Clusterzahl.
heights_desc = np.sort(Z_full[:, 2])[::-1]
ward_merge = [heights_desc[k - 2] for k in ward_k]   # Höhe, um k Cluster zu bilden

ward_metrics = pd.DataFrame({"k": list(ward_k), "Merge_Distanz": ward_merge,
                             "Silhouette": w_sil, "Davies_Bouldin": w_dbi}).set_index("k")

w_best_sil = ward_metrics["Silhouette"].idxmax()
w_best_dbi = ward_metrics["Davies_Bouldin"].idxmin()

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(list(ward_k), ward_merge, "o-", color="#4C72B0"); ax[0].set_title("Merge-Distanz – Knick suchen")
ax[1].plot(list(ward_k), w_sil, "o-", color="#55A868");     ax[1].set_title("Silhouette – höher = besser")
ax[2].plot(list(ward_k), w_dbi, "o-", color="#C44E52");     ax[2].set_title("Davies-Bouldin – niedriger = besser")

ax[1].axvline(w_best_sil, ls="--", c="grey", alpha=.7)
ax[1].annotate(f"max @ k={w_best_sil}", (w_best_sil, ward_metrics.loc[w_best_sil, "Silhouette"]),
               textcoords="offset points", xytext=(6, -10), fontsize=8)
ax[2].axvline(w_best_dbi, ls="--", c="grey", alpha=.7)
ax[2].annotate(f"min @ k={w_best_dbi}", (w_best_dbi, ward_metrics.loc[w_best_dbi, "Davies_Bouldin"]),
               textcoords="offset points", xytext=(6, 6), fontsize=8)

for a in ax:
    a.set_xlabel("k")
plt.tight_layout(); plt.show()

print(f"Ward-Optimum: Silhouette bei k={w_best_sil}, Davies-Bouldin bei k={w_best_dbi}")
ward_metrics.T.round(3)

In [ ]:
from scipy.cluster.hierarchy import fcluster

# Ward-Linkage EINMAL berechnen (wiederverwenden, falls aus dem Ward-Scan schon da)
try:
    Z_full
except NameError:
    Z_full = linkage(X, method="ward")

ks = list(range(2, 12))
ncols = 3
nrows = int(np.ceil(len(ks) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.6 * nrows))
axes = axes.ravel()

for ax, k in zip(axes, ks):
    lab = fcluster(Z_full, t=k, criterion="maxclust")          # k Cluster aus dem Baum schneiden
    sil = silhouette_score(X, lab, sample_size=min(SIL_SAMPLE, len(X)),
                           random_state=RANDOM_STATE)
    plot_clusters(lab, f"Ward k={k}  (Sil={sil:.3f})", emb=X_pca, ax=ax)

for ax in axes[len(ks):]:
    ax.set_visible(False)

fig.suptitle("Ward (hierarchisch) auf X (PCA-2D-Projektion), k = 2 … 11",
             fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

### 5.3 · DBSCAN
`eps` bestimmen wir über die **k-Distance-Kurve** (Distanz zum `min_samples`-ten Nachbarn, sortiert). Der Knick markiert eine sinnvolle Nachbarschafts-Größe. Faustregel `min_samples ≈ 2 · Anzahl Features`.

In [ ]:
from sklearn.neighbors import NearestNeighbors

# Helfer: eps am Knie der k-Distance-Kurve.
# Die Kurve ist konvex (flach -> steiler Anstieg), liegt also UNTER der Sehne.
# Das Knie = Punkt mit maximalem Abstand UNTER der Sehne -> argmax(line - kd).
def knee_eps(X, ms):
    nn = NearestNeighbors(n_neighbors=ms).fit(X)
    kd = np.sort(nn.kneighbors(X)[0][:, -1])
    line = np.linspace(kd[0], kd[-1], len(kd))
    eps = kd[np.argmax(line - kd)]
    return eps, kd

min_samples = 2 * X.shape[1]                 # Faustregel 2*D = 12  (D = 6 Features)
eps, kdist = knee_eps(X, min_samples)
print(f"D = {X.shape[1]},  min_samples = {min_samples},  gewähltes eps = {eps:.3f}")

plt.figure(figsize=(7, 4))
plt.plot(np.arange(len(kdist)), kdist, color="#4C72B0")
plt.axhline(eps, ls="--", c="red", label=f"eps = {eps:.2f}")
plt.xlabel("sortierte Punkte"); plt.ylabel(f"Distanz zum {min_samples}. Nachbarn")
plt.title("k-Distance-Plot (DBSCAN eps-Bestimmung)"); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
db = DBSCAN(eps=eps, min_samples=min_samples).fit(X)
labels_db = evaluate("DBSCAN", db.labels_)
plot_clusters(labels_db, f"DBSCAN (eps={eps:.2f}, min_samples={min_samples})")

In [ ]:
# Hängt das Ergebnis an min_samples? Faustregel 2*D=12 vs. Untergrenze D+1=7,
# je einmal mit festem eps (isoliert min_samples) und mit neu abgeleitetem eps.
ms_rule = 2 * X.shape[1]       # 12
ms_min  = X.shape[1] + 1       # 7
eps_ref, _ = knee_eps(X, ms_rule)

def dbscan_eval(ms, eps_val):
    lab = DBSCAN(eps=eps_val, min_samples=ms).fit(X).labels_
    nc = len(set(lab)) - (1 if -1 in lab else 0)
    noise = int((lab == -1).sum())
    return nc, noise, round(noise / len(X) * 100, 1)

rows = []
for ms in (ms_rule, ms_min):
    for src, e in [("eps fest (Regel)", eps_ref), ("eps neu abgeleitet", knee_eps(X, ms)[0])]:
        nc, no, pc = dbscan_eval(ms, e)
        rows.append({"min_samples": ms, "eps": round(float(e), 3), "eps_Quelle": src,
                     "Cluster": nc, "Noise": no, "Noise_%": pc})
sens = pd.DataFrame(rows)
display(sens)

nl = chr(10)
labels_bar = [f"ms={r.min_samples}{nl}{r.eps_Quelle}" for r in sens.itertuples()]
plt.figure(figsize=(8, 3.6))
bars = plt.bar(labels_bar, sens["Noise_%"],
               color=["#C44E52", "#C44E52", "#4C72B0", "#4C72B0"])
plt.ylabel("Noise-Anteil (%)"); plt.ylim(0, 100)
plt.title("DBSCAN: viele kleine Dichtezellen statt weniger Segmente")
for b, v in zip(bars, sens["Noise_%"]):
    plt.text(b.get_x() + b.get_width() / 2, v + 1, f"{v:.1f}%", ha="center", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
def plot_dbscan_panel(ms, eps_val, ax):
    lab = DBSCAN(eps=eps_val, min_samples=ms).fit(X).labels_
    noise = lab == -1
    nc = len(set(lab)) - (1 if -1 in lab else 0)
    ax.scatter(X_pca[noise, 0], X_pca[noise, 1], c="lightgrey", s=5, alpha=.30, linewidths=0)
    ax.scatter(X_pca[~noise, 0], X_pca[~noise, 1], c=lab[~noise], cmap="tab20",
               s=9, alpha=.75, linewidths=0)
    ax.set_title(f"min_samples={ms}:  {nc} Cluster,  {noise.mean()*100:.1f}% Noise")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
plot_dbscan_panel(ms_rule, eps_ref, axes[0])   # Faustregel 12
plot_dbscan_panel(ms_min,  eps_ref, axes[1])   # Untergrenze 7, gleiches eps
fig.suptitle("DBSCAN: Effekt von min_samples bei gleichem eps  (grau = Noise)",
             fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# =============================================================================
# "k = 2..10 bei DBSCAN?" - DBSCAN hat KEIN k als Parameter! Die Clusterzahl ist
# ein ERGEBNIS aus (eps, min_samples). Wir drehen an eps, bis DBSCAN moeglichst
# genau die Zielzahl liefert; min_samples bleibt fest = 2*n_features.
# KEIN evaluate() -> Haupt-Vergleichstabelle (results) bleibt unberuehrt.
# =============================================================================
def _sil_X_db(labels):
    """Silhouette im echten Feature-Raum X (Noise -1 ausgeschlossen)."""
    labels = np.asarray(labels); m = labels != -1
    if len(set(labels[m].tolist())) < 2:
        return np.nan
    try:
        return silhouette_score(X[m], labels[m],
                                sample_size=min(SIL_SAMPLE, int(m.sum())),
                                random_state=RANDOM_STATE)
    except ValueError:
        return np.nan

def _largest_share(labels):
    labels = np.asarray(labels); nn = labels[labels != -1]
    return 100 * np.bincount(nn).max() / len(labels) if len(nn) else 0.0

_ms_db = 2 * X.shape[1]                         # = 12

# eps-Raster aus der k-Distance-Verteilung: klein (Mikro-Cluster) bis gross (k=1)
_nn_db = NearestNeighbors(n_neighbors=_ms_db).fit(X)
_kd_db = np.sort(_nn_db.kneighbors(X)[0][:, -1])
eps_scan = np.linspace(np.percentile(_kd_db, 30), np.percentile(_kd_db, 99.5), 45)

# ---- db_runs AUFBAUEN: je eps einmal DBSCAN, k/Noise/Silhouette merken ----
db_runs = []
for e in eps_scan:
    lab = DBSCAN(eps=float(e), min_samples=_ms_db).fit(X).labels_
    db_runs.append({"eps": float(e),
                    "k": len(set(lab.tolist()) - {-1}),
                    "noise": float((lab == -1).mean()),
                    "sil": _sil_X_db(lab)})
db_runs = pd.DataFrame(db_runs)

# ---- Fuer jede Ziel-Clusterzahl die naechstbeste eps-Konfiguration waehlen ----
rows = []
for target in range(2, 11):
    cand = db_runs.assign(dist=(db_runs["k"] - target).abs())
    best = cand.sort_values(["dist", "noise"]).iloc[0]
    lab_b = DBSCAN(eps=float(best["eps"]), min_samples=_ms_db).fit(X).labels_
    rows.append({"Ziel_k": target,
                 "erreichtes_k": int(best["k"]),
                 "getroffen": "ja" if int(best["k"]) == target else "nein",
                 "eps": round(float(best["eps"]), 3),
                 "min_samples": _ms_db,
                 "Noise_%": round(best["noise"] * 100, 1),
                 "größtes_Cl_%": round(_largest_share(lab_b), 1),
                 "Silhouette_auf_X": round(best["sil"], 3) if pd.notna(best["sil"]) else np.nan})
dbscan_k_table = pd.DataFrame(rows).set_index("Ziel_k")

# ---- Visualisierung ----
tk = dbscan_k_table.index.to_numpy()
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
ax[0].plot(tk, tk, "--", color="grey", lw=1, label="ideal (erreicht = Ziel)")
ax[0].plot(tk, dbscan_k_table["erreichtes_k"], "o-", color="#4C72B0", label="tatsaechlich erreicht")
ax[0].set(title="Erreichte vs. gewuenschte Clusterzahl", xlabel="Ziel-k", ylabel="erreichtes k")
ax[0].legend(fontsize=8)
ax[1].plot(tk, dbscan_k_table["größtes_Cl_%"], "o-", color="#C44E52")
ax[1].axhline(90, color="grey", ls=":", lw=1)
ax[1].set(title="Größtes Cluster (% aller Punkte)", xlabel="Ziel-k", ylabel="%")
ax[2].axhline(0, color="black", lw=.7)
ax[2].plot(tk, dbscan_k_table["Silhouette_auf_X"], "o-", color="#55A868")
ax[2].set(title="Silhouette (auf X) der DBSCAN-Loesung", xlabel="Ziel-k", ylabel="Silhouette")
fig.suptitle("DBSCAN 'auf k=2..10 gebracht': eps getunt, min_samples fix = %d" % _ms_db,
             fontsize=14, y=1.03)
plt.tight_layout(); plt.show()

print("DBSCAN auf Ziel-Clusterzahl gebracht (min_samples fix = %d):" % _ms_db)
print(dbscan_k_table)
print("\nLesart: DBSCAN kennt kein k - die Zielzahl wird nur durch eps-Drehen angenaehert.")
print("Wo eine positive Silhouette steht, schluckt EIN Cluster ~99% der Punkte (Spalte größtes_Cl_%);")
print("die uebrigen 'Cluster' sind nur abgespaltene Ausreisser-Pockets, keine echten Segmente.")
print("-> DBSCAN findet keine ausgewogene Mehr-Segment-Partition: der Markt ist ein Kontinuum.")

In [ ]:
_ms_db = 2 * X.shape[1]
for target in range(2, 11):
    cand = db_runs.assign(dist=(db_runs["k"] - target).abs())
    best = cand.sort_values(["dist", "noise"]).iloc[0]
    lab_b = DBSCAN(eps=float(best["eps"]), min_samples=_ms_db).fit(X).labels_
    rows.append({"Ziel_k": target, "erreichtes_k": int(best["k"]),
                 "getroffen": "ja" if int(best["k"]) == target else "nein",
                 "eps": round(float(best["eps"]), 3), "min_samples": _ms_db,
                 "Noise_%": round(best["noise"] * 100, 1),
                 "größtes_Cl_%": round(_largest_share(lab_b), 1),     # NEU
                 "Silhouette_auf_X": round(best["sil"], 3) if pd.notna(best["sil"]) else np.nan})

### 5.4 · HDBSCAN
Erweitert DBSCAN über verschiedene Dichte-Skalen und braucht kein festes `eps`. Wir steuern nur die minimale Clustergröße.

In [ ]:
hdb = HDBSCAN(min_cluster_size=250, min_samples=15, copy=True).fit(X)
labels_hdb = evaluate("HDBSCAN", hdb.labels_)
plot_clusters(labels_hdb, "HDBSCAN")

### 5.6 · t-SNE-Projektion (nichtlinear, nur zur Visualisierung)
PC1/PC2 fangen nur einen Teil der Varianz ein, weshalb die Cluster im PCA-Bild stark überlappen. **t-SNE** ist eine nichtlineare Projektion, die lokale Nachbarschaften erhält und die Segmente optisch meist deutlich klarer trennt.

Wie die PCA dient t-SNE **nur der Visualisierung** – geclustert wird weiterhin auf allen 8 Features. Wichtig: t-SNE-Achsen und -Abstände sind **nicht** quantitativ interpretierbar (keine feste Bedeutung, kein „größer = teurer"), sie zeigen nur, *ob* sich Gruppen sauber separieren lassen.

In [ ]:
from sklearn.manifold import TSNE

# t-SNE sicherstellen (EINMAL ~30-40 s; wird wiederverwendet, falls schon da)
try:
    X_tsne, tsne_idx
except NameError:
    tsne_idx = np.arange(len(X))
    X_tsne = TSNE(n_components=2, perplexity=30, init="pca",
                  learning_rate="auto", random_state=RANDOM_STATE).fit_transform(X[tsne_idx])
    print("t-SNE berechnet für", len(tsne_idx), "Punkte")

# Nur Verfahren aufnehmen, deren Labels wirklich existieren
_wanted = [("K-Means", "labels_km"), ("Hierarchical", "labels_agg"), ("DBSCAN", "labels_db"),
           ("HDBSCAN", "labels_hdb"), ("GMM", "labels_gmm")]
runs = [(name, globals()[v]) for name, v in _wanted if v in globals()]
print("Enthaltene Verfahren:", [n for n, _ in runs])

ncols = 3
nrows = int(np.ceil(len(runs) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5.5 * nrows))
axes = axes.ravel()
for ax, (name, lab) in zip(axes, runs):
    plot_clusters(lab, name, emb=X_tsne, idx=tsne_idx, ax=ax,
                  xlabel="t-SNE 1", ylabel="t-SNE 2")
for ax in axes[len(runs):]:
    ax.set_visible(False)
fig.suptitle(f"Cluster im t-SNE-Raum (alle n={len(tsne_idx):,} Objekte)", fontsize=15, y=1.0)
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.manifold import TSNE

def tsne_kgrid(label_of_k, ks, title, ncols=3):
    """Zeigt eine Clusterlösung je k im t-SNE-Raum. label_of_k(k) -> labels."""
    global X_tsne, tsne_idx
    if "X_tsne" not in globals():
        tsne_idx = np.arange(len(X))
        X_tsne = TSNE(n_components=2, perplexity=30, init="pca",
                      learning_rate="auto", random_state=RANDOM_STATE).fit_transform(X[tsne_idx])
        print("t-SNE berechnet für", len(tsne_idx), "Punkte")
    nrows = int(np.ceil(len(ks) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.6 * nrows))
    axes = axes.ravel()
    for ax, k in zip(axes, ks):
        lab = label_of_k(k)
        sil = silhouette_score(X, lab, sample_size=min(SIL_SAMPLE, len(X)), random_state=RANDOM_STATE)
        plot_clusters(lab, f"k={k}  (Sil={sil:.3f})", emb=X_tsne, idx=tsne_idx, ax=ax,
                      xlabel="t-SNE 1", ylabel="t-SNE 2")
    for ax in axes[len(ks):]:
        ax.set_visible(False)
    fig.suptitle(title, fontsize=16, fontweight="bold")
    plt.tight_layout(); plt.show()

In [ ]:
tsne_kgrid(lambda k: KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(X),
           list(range(2, 10)), "K-Means im t-SNE-Raum, k = 2 … 9")

In [ ]:
from scipy.cluster.hierarchy import fcluster

if "Z_full" not in globals():                  # Ward-Linkage einmal (teuer, wiederverwenden)
    Z_full = linkage(X, method="ward")

tsne_kgrid(lambda k: fcluster(Z_full, t=k, criterion="maxclust"),
           list(range(2, 10)), "Ward (hierarchisch) im t-SNE-Raum, k = 2 … 9")

### 5.7 · UMAP-Projektion (Vergleich zu t-SNE)
**UMAP** ist eine zweite nichtlineare Projektion. Gegenüber t-SNE erhält es neben der lokalen auch die **globale** Struktur meist besser (Abstände zwischen Clustern sind etwas aussagekräftiger) und ist schneller. Wir rechnen es auf **denselben Punkten** (allen 21.420) wie t-SNE, damit der Vergleich fair ist.

Auch UMAP dient **nur der Visualisierung** – geclustert wird weiter auf allen 8 Features, und die Achsenwerte sind nicht quantitativ interpretierbar. Zeigen beide Projektionen (t-SNE *und* UMAP) dieselben Gruppen, ist das ein starkes Indiz, dass die Segmente **real** sind und kein Projektionsartefakt.

In [ ]:
def emb_kgrid(emb, idx, label_of_k, ks, title, xlabel="Dim 1", ylabel="Dim 2", ncols=4):
    """Zeigt je k eine Clusterlösung in einer beliebigen 2D-Projektion emb."""
    nrows = int(np.ceil(len(ks) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.6 * nrows))
    axes = axes.ravel()
    for ax, k in zip(axes, ks):
        lab = label_of_k(k)
        sil = silhouette_score(X, lab, sample_size=min(SIL_SAMPLE, len(X)), random_state=RANDOM_STATE)
        plot_clusters(lab, f"k={k}  (Sil={sil:.3f})", emb=emb, idx=idx, ax=ax,
                      xlabel=xlabel, ylabel=ylabel)
    for ax in axes[len(ks):]:
        ax.set_visible(False)
    fig.suptitle(title, fontsize=16, fontweight="bold")
    plt.tight_layout(); plt.show()

In [ ]:
assert "X_umap" in globals(), "Erst die UMAP-Zelle laufen lassen (X_umap fehlt)."
ks = list(range(2, 10))
emb_kgrid(X_umap, tsne_idx,
          lambda k: KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(X),
          ks, "K-Means im UMAP-Raum, k = 2 … 9", xlabel="UMAP 1", ylabel="UMAP 2")

In [ ]:
from scipy.cluster.hierarchy import fcluster
if "Z_full" not in globals():
    Z_full = linkage(X, method="ward")
emb_kgrid(X_umap, tsne_idx,
          lambda k: fcluster(Z_full, t=k, criterion="maxclust"),
          ks, "Ward im UMAP-Raum, k = 2 … 9", xlabel="UMAP 1", ylabel="UMAP 2")

In [ ]:
import warnings
import numpy, umap
print("NumPy:", numpy.__version__, "| umap:", umap.__version__)

# gleiche Punkte wie t-SNE (fairer Vergleich); falls tsne_idx fehlt: alle Punkte
if "tsne_idx" not in globals():
    tsne_idx = np.arange(len(X))

# UMAP (random_state -> reproduzierbar, aber single-threaded => etwas langsamer, ~40-60 s)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    X_umap = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2,
                       random_state=RANDOM_STATE).fit_transform(X[tsne_idx])
print("UMAP berechnet für", len(tsne_idx), "Punkte")

# Referenz-Clustering + k für die Titel (robust, falls K/labels_km beim Testen fehlen)
kref = globals().get("K", 4)
lab_ref = labels_km if "labels_km" in globals() else \
          KMeans(n_clusters=kref, n_init=10, random_state=RANDOM_STATE).fit_predict(X)

# t-SNE sicherstellen (für den Direktvergleich in a)
if "X_tsne" not in globals():
    from sklearn.manifold import TSNE
    X_tsne = TSNE(n_components=2, perplexity=30, init="pca",
                  learning_rate="auto", random_state=RANDOM_STATE).fit_transform(X[tsne_idx])

# a) t-SNE vs. UMAP am selben Clustering
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
plot_clusters(lab_ref, "K-Means · t-SNE", emb=X_tsne, idx=tsne_idx, ax=axes[0],
              xlabel="t-SNE 1", ylabel="t-SNE 2")
plot_clusters(lab_ref, "K-Means · UMAP", emb=X_umap, idx=tsne_idx, ax=axes[1],
              xlabel="UMAP 1", ylabel="UMAP 2")
fig.suptitle(f"Projektions-Vergleich: t-SNE vs. UMAP (gleiche Punkte, K-Means k={kref})",
             fontsize=15, y=1.02)
plt.tight_layout(); plt.show()

# b) alle VORHANDENEN Verfahren im UMAP-Raum
_wanted = [("K-Means", "labels_km"), ("Hierarchical", "labels_agg"), ("DBSCAN", "labels_db"),
           ("HDBSCAN", "labels_hdb"), ("GMM", "labels_gmm")]
runs = [(n, globals()[v]) for n, v in _wanted if v in globals()]
ncols = 3
nrows = int(np.ceil(len(runs) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5.5 * nrows))
axes = axes.ravel()
for ax, (name, lab) in zip(axes, runs):
    plot_clusters(lab, name, emb=X_umap, idx=tsne_idx, ax=ax, xlabel="UMAP 1", ylabel="UMAP 2")
for ax in axes[len(runs):]:
    ax.set_visible(False)
fig.suptitle(f"Cluster im UMAP-Raum (alle n={len(tsne_idx):,} Objekte)", fontsize=15, y=1.0)
plt.tight_layout(); plt.show()

### 5.8 · HDBSCAN gezielt auf ~4 Cluster gebracht (auf X vs. auf dem Embedding)


In [ ]:
TARGET_K = 4      # Ziel-Clusterzahl zentral (statt an mehreren Stellen hardcoded)

def hdbscan_to_k(data, target=TARGET_K, min_samples=10,
                 sizes=(250, 400, 600, 900, 1400, 1600, 2000, 3000)):
    """Sucht die min_cluster_size, deren HDBSCAN-Ergebnis der Ziel-Clusterzahl am
    naechsten kommt (Tie-Break: weniger Noise). HDBSCAN waehlt selbst, WIE VIELE
    Cluster; wir steuern ueber die minimale Clustergroesse nur die Aufloesung."""
    runs = []
    for mcs in sizes:
        lab = HDBSCAN(min_cluster_size=int(mcs), min_samples=min_samples,
                      copy=True).fit(data).labels_          # copy=True -> keine FutureWarning
        runs.append({"mcs": int(mcs), "k": len(set(lab) - {-1}),
                     "noise": float((lab == -1).mean()), "labels": lab})
    runs.sort(key=lambda r: (abs(r["k"] - target), r["noise"]))
    return runs[0], runs

def sil_on_X(labels):
    """Silhouette IMMER im echten Feature-Raum X (fairer, vergleichbarer Massstab),
    egal worauf geclustert wurde. Noise (-1) ausgeschlossen."""
    labels = np.asarray(labels); m = labels != -1
    if len(set(labels[m].tolist())) < 2:
        return np.nan
    return silhouette_score(X[m], labels[m],
                            sample_size=min(SIL_SAMPLE, int(m.sum())),
                            random_state=RANDOM_STATE)

def largest_share(labels):
    labels = np.asarray(labels); nn = labels[labels != -1]
    return round(100 * np.bincount(nn).max() / len(labels), 1) if len(nn) else np.nan

# Varianten: A auf X; B auf den 2D-Projektionen (nur, wenn vorhanden -> kein NameError)
variants = [("A_X", f"A · HDBSCAN auf X ({X.shape[1]} Feat.)", X)]
if "X_tsne" in globals(): variants.append(("B_tsne", "B · HDBSCAN auf t-SNE (2D)", X_tsne))
if "X_umap" in globals(): variants.append(("B_umap", "B · HDBSCAN auf UMAP (2D)", X_umap))

rows, labels_by = [], {}
for tag, name, data in variants:
    best, _ = hdbscan_to_k(data, target=TARGET_K)
    labels_by[tag] = best["labels"]
    rows.append({"Variante": name, "min_cluster_size": best["mcs"],
                 "n_Cluster": best["k"],
                 "getroffen": "ja" if best["k"] == TARGET_K else "nein",
                 "Noise_%": round(best["noise"] * 100, 1),
                 "größtes_Cl_%": largest_share(best["labels"]),
                 "Silhouette_auf_X": round(sil_on_X(best["labels"]), 3)})
hdb_summary = pd.DataFrame(rows).set_index("Variante")

# Labels fuer spaetere Plots bereitstellen
labels_hdb_X    = labels_by.get("A_X")
labels_hdb_tsne = labels_by.get("B_tsne")
labels_hdb_umap = labels_by.get("B_umap")

print("Ziel-Clusterzahl:", TARGET_K)
display(hdb_summary)
print(f"\nSilhouette bei ALLEN Varianten im echten {X.shape[1]}-Feature-Raum X gemessen,")
print("damit A (auf X) und B (auf 2D-Projektion) vergleichbar sind.")
print("'größtes_Cl_%' entlarvt, ob eine Lösung nur ein Mega-Cluster + Reste ist.")

In [ ]:
# HDBSCAN "auf k=2..9 gebracht": min_cluster_size variieren (k ist ein ERGEBNIS).
# Sweep EINMAL ueber ein mcs-Raster, dann je Ziel-k die naechste Loesung waehlen.
# Nutzt sil_on_X() und largest_share() aus der vorherigen Zelle.
mcs_grid = [100, 150, 200, 300, 400, 600, 800, 1100, 1500, 2000, 2800, 4000, 6000]
min_samples = 10

sweep = []
for mcs in mcs_grid:
    lab = HDBSCAN(min_cluster_size=int(mcs), min_samples=min_samples, copy=True).fit(X).labels_
    sweep.append({"mcs": int(mcs), "k": len(set(lab) - {-1}),
                  "noise": float((lab == -1).mean()), "labels": lab})

rows = []
for target in range(2, 40):
    best = sorted(sweep, key=lambda r: (abs(r["k"] - target), r["noise"]))[0]
    rows.append({"Ziel_k": target, "erreichtes_k": best["k"],
                 "getroffen": "ja" if best["k"] == target else "nein",
                 "min_cluster_size": best["mcs"],
                 "Noise_%": round(best["noise"] * 100, 1),
                 "größtes_Cl_%": largest_share(best["labels"]),
                 "Silhouette_auf_X": sil_on_X(best["labels"])})
hdb_k_table = pd.DataFrame(rows).set_index("Ziel_k")
display(hdb_k_table)

tk = hdb_k_table.index.to_numpy()
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
ax[0].plot(tk, tk, "--", color="grey", lw=1, label="ideal (erreicht = Ziel)")
ax[0].plot(tk, hdb_k_table["erreichtes_k"], "o-", color="#4C72B0", label="erreicht")
ax[0].set(title="Erreichte vs. gewünschte Clusterzahl", xlabel="Ziel-k", ylabel="erreichtes k"); ax[0].legend(fontsize=8)
ax[1].plot(tk, hdb_k_table["größtes_Cl_%"], "o-", color="#C44E52"); ax[1].axhline(90, color="grey", ls=":", lw=1)
ax[1].set(title="Größtes Cluster (% aller Punkte)", xlabel="Ziel-k", ylabel="%")
ax[2].axhline(0, color="black", lw=.7)
ax[2].plot(tk, hdb_k_table["Silhouette_auf_X"], "o-", color="#55A868")
ax[2].set(title="Silhouette (auf X)", xlabel="Ziel-k", ylabel="Silhouette")
fig.suptitle("HDBSCAN 'auf k=2..9 gebracht' (min_cluster_size getunt, min_samples=%d)" % min_samples,
             fontsize=14, y=1.03)
plt.tight_layout(); plt.show()

## 6 · Bewertung & Vergleich der Algorithmen
 Silhouette (höher = besser) und Davies-Bouldin (niedriger = besser). Noise = als Ausreißer markierte Punkte.
 Für K-Means, Ward und GMM wird hier zunächst das finale Modell mit k=4 gefittet und mit in die Vergleichstabelle aufgenommen (die vier Hypothesen H1–H4 geben die Segmentzahl fachlich vor).

In [ ]:
# Finale Clusterings sicherstellen: §5.1/§5.2 scannen nur über k, ohne ein finales
# Modell zu fixieren -> hier einmal mit k=4 fitten und per evaluate() in die
# Vergleichstabelle aufnehmen (Kennzahlen aus §3 favorisieren knapp k=3, die vier
# Hypothesen H1-H4 geben aber vier Segmente vor). GMM als fünftes Verfahren:
# probabilistisches Pendant zu K-Means (Import aus §1).
from scipy.cluster.hierarchy import fcluster

K = 4
_done = {r["Algorithmus"] for r in results}
if "K-Means" not in _done:
    labels_km = evaluate("K-Means", KMeans(n_clusters=K, n_init=10,
                                           random_state=RANDOM_STATE).fit_predict(X))
if "Hierarchical" not in _done:
    if "Z_full" not in globals():
        Z_full = linkage(X, method="ward")
    labels_agg = evaluate("Hierarchical", fcluster(Z_full, t=K, criterion="maxclust"))
if "GMM" not in _done:
    labels_gmm = evaluate("GMM", GaussianMixture(n_components=K, n_init=5,
                                                 random_state=RANDOM_STATE).fit_predict(X))

comparison = pd.DataFrame(results).set_index("Algorithmus")
comparison = comparison.sort_values("Silhouette", ascending=False)
comparison.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
comparison["Silhouette"].plot.bar(ax=ax[0], color="#55A868")
ax[0].set_title("Silhouette (höher = besser)"); ax[0].tick_params(axis="x", rotation=30)
comparison["Davies_Bouldin"].plot.bar(ax=ax[1], color="#C44E52")
ax[1].set_title("Davies-Bouldin (niedriger = besser)"); ax[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

### 6.1 · Auswahl & Visualisierung der zwei besten Clusterings


In [ ]:
# --- Auswahl der ZWEI BESTEN Clusterings ---
comp = pd.DataFrame(results).set_index("Algorithmus")
comp["Noise_%"] = (comp["Noise"] / len(X) * 100).round(1)

# Für eine Käufer-SEGMENTIERUNG muss (fast) jeder Punkt einem Segment gehören.
# Verfahren, die mehr als 30 % der Punkte als Noise abstempeln, wären keine
# vollständige Segmentierung und werden ausgeschlossen. DBSCAN/HDBSCAN bleiben
# hier mit ~5-7 % Noise zwar im Rennen, ranken aber über die Silhouette klar hinter
# den Partitionsverfahren.
MAX_NOISE_FRAC = 0.30
valid = comp[comp["Noise"] / len(X) <= MAX_NOISE_FRAC]
ranked = valid.sort_values(["Silhouette", "Davies_Bouldin"], ascending=[False, True])
best_names = list(ranked.head(2).index)

print("Vollständige Segmentierungen (Noise <= 30 %), Rangfolge nach Silhouette:")
print(ranked[["n_Cluster", "Noise_%", "Silhouette", "Davies_Bouldin"]].round(3))
print("\nAusgeschlossen (zu viel Noise -> keine echte Segmentierung):",
      list(comp.index.difference(valid.index)))
print(">> Die zwei besten Clusterings:", best_names)

# --- Genau diese zwei in t-SNE UND UMAP visualisieren (Deliverable 3) ---
# Nur Labels aufnehmen, die wirklich existieren (gleiches Muster wie §5.6/§5.7)
_wanted = [("K-Means", "labels_km"), ("Hierarchical", "labels_agg"), ("DBSCAN", "labels_db"),
           ("HDBSCAN", "labels_hdb"), ("GMM", "labels_gmm")]
label_map = {name: globals()[v] for name, v in _wanted if v in globals()}
assert "X_tsne" in globals() and "X_umap" in globals(), \
    "Erst §5.6 (t-SNE) und §5.7 (UMAP) laufen lassen."

fig, axes = plt.subplots(len(best_names), 2, figsize=(13, 6 * len(best_names)))
axes = np.atleast_2d(axes)
for r, name in enumerate(best_names):
    plot_clusters(label_map[name], f"{name} · t-SNE", emb=X_tsne, idx=tsne_idx,
                  ax=axes[r, 0], xlabel="t-SNE 1", ylabel="t-SNE 2")
    plot_clusters(label_map[name], f"{name} · UMAP", emb=X_umap, idx=tsne_idx,
                  ax=axes[r, 1], xlabel="UMAP 1", ylabel="UMAP 2")
fig.suptitle(f"Die zwei besten Clusterings in t-SNE & UMAP: {best_names[0]} & {best_names[1]}",
             fontsize=15, y=1.0)
plt.tight_layout(); plt.show()

## 7 · Cluster-Profile & Abgleich mit den Hypothesen


In [ ]:
prof = df.copy()
prof["cluster"] = labels_km

# Profil A: die 6 standardisierten Cluster-Features (Basis der Segmentierung)
profile = prof.groupby("cluster")[clustering_cols].mean()
sizes = prof["cluster"].value_counts().sort_index()
profile.insert(0, "n", sizes)
display(profile.round(2))

# Profil B: externe Validierung über die prof_-Spalten (im Clustering NICHT verwendet).
# Trennen sich Preis/Grade/Waterfront trotzdem klar zwischen den Clustern,
# sind die Segmente auch ökonomisch echt - nicht nur geometrisch.
profile_ext = prof.groupby("cluster")[["prof_price", "prof_waterfront", "prof_grade"]].mean()
profile_ext.insert(0, "n", sizes)
profile_ext.round({"prof_price": 0, "prof_waterfront": 3, "prof_grade": 2})

In [ ]:
# Interpretation je Cluster (aus den Profil-Mittelwerten oben abgeleitet, k=4).
# Deterministisch bei RANDOM_STATE=20; falls sich die Cluster-Nummern bei einem
# erneuten Lauf verschieben, hier einmal anpassen.
segment_names = {
    0: "Luxus (H3)",
    1: "Urban kompakt (H2)",
    2: "Erstkäufer Süd (H4)",
    3: "Familien Umland (H1)",
}

hm = profile.drop(columns="n").T          # Features = Zeilen (y), Cluster = Spalten (x)
xlabels = [f"Cluster {c}\n{segment_names.get(c, '?')}\n(n={int(profile.loc[c, 'n']):,})"
           for c in hm.columns]

plt.figure(figsize=(10, 4.2))
ax = sns.heatmap(hm, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
                 cbar_kws={"shrink": .8})
ax.set_xticklabels(xlabels, rotation=0, fontsize=8)
plt.title("Cluster-Profile & Segment-Zuordnung (Ø standardisierte Features je Cluster, k=4)")
plt.xlabel("Cluster / Segment"); plt.ylabel("Feature")
plt.tight_layout(); plt.show()

### Klarere Segment-Darstellung: Snake-Plot & Radar-Charts
Da der t-SNE/UMAP-Scatter durch die Überlappung unscharf ist, zeigen wir die Trennschärfe über die **Cluster-Mittelwerte** – das macht die Segmente eindeutig sichtbar.

In [ ]:
# Snake-Plot: Profil-Linie je Cluster über alle Features (Klassiker der Segmentierung).
# Zeigt die Trennschärfe der Segmente viel klarer als der überlappende t-SNE/UMAP-Scatter,
# weil hier die CLUSTER-MITTELWERTE verglichen werden, nicht die Einzelpunkte.
prof_only = profile.drop(columns="n")
cmap = plt.get_cmap("tab10")

plt.figure(figsize=(11, 4.8))
for c in prof_only.index:
    plt.plot(prof_only.columns, prof_only.loc[c], "o-", linewidth=2, color=cmap(c % 10),
             label=f"Cluster {c} - {segment_names.get(c, '?')} (n={int(profile.loc[c, 'n']):,})")
plt.axhline(0, color="grey", lw=1, ls="--")           # 0 = Gesamtdurchschnitt
plt.xticks(rotation=30, ha="right")
plt.ylabel("Ø standardisierter Wert  (+ ueber / - unter Durchschnitt)")
plt.title("Snake-Plot: Segment-Profile je Cluster (K-Means, k=4)")
plt.legend(fontsize=8, loc="best")
plt.tight_layout(); plt.show()


In [ ]:
# Radar-Charts: ein Netz je Segment (Persona-Darstellung).
# Gleiche radiale Skala fuer alle vier -> die Formen sind direkt vergleichbar.
feats = list(profile.drop(columns="n").columns)
N = len(feats)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]                      # Kreis schliessen

vals_all = profile.drop(columns="n").values
gmin, gmax = float(vals_all.min()) - 0.3, float(vals_all.max()) + 0.3
cmap = plt.get_cmap("tab10")

fig, axes = plt.subplots(2, 2, figsize=(12, 11), subplot_kw=dict(polar=True))
for ax, c in zip(axes.ravel(), profile.index):
    vals = profile.drop(columns="n").loc[c].tolist()
    vals += vals[:1]
    ax.plot(angles, vals, "o-", linewidth=2, color=cmap(c % 10))
    ax.fill(angles, vals, color=cmap(c % 10), alpha=0.25)
    ax.plot(angles, [0] * len(angles), color="grey", lw=1, ls="--")   # 0 = Durchschnitt
    ax.set_ylim(gmin, gmax)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(feats, fontsize=7)
    ax.set_yticklabels([])
    ax.set_title(f"Cluster {c} - {segment_names.get(c, '?')}  (n={int(profile.loc[c, 'n']):,})",
                 fontsize=11, pad=22)
fig.suptitle("Segment-Steckbriefe als Radar-Charts (Ø standardisierte Features, k=4)",
             fontsize=14, y=1.0)
plt.tight_layout(); plt.show()
